In [ ]:
# =====================================
# PATCHTST MODEL FOR PAMAP2 DATASET 
# =====================================
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, f1_score, precision_score, 
                           recall_score, roc_auc_score, mean_absolute_error,
                           mean_squared_error, r2_score, confusion_matrix,
                           classification_report)
import time
import psutil
import os
from datetime import datetime
import math

# Force CPU for stability
device = torch.device('cpu')
print("Using CPU for stability")

# =====================================
# 1. Data Loading 
# =====================================
def load_pamap2_data(data_path="PAMAP2_Dataset"):
    print("Loading PAMAP2 dataset from local files...")
    all_data = []
    all_labels = []
    
    column_names = ['timestamp', 'activity_id', 'heart_rate']
    imu_columns = [
        'temperature', 'acceleration_16g_x', 'acceleration_16g_y', 'acceleration_16g_z',
        'acceleration_6g_x', 'acceleration_6g_y', 'acceleration_6g_z',
        'gyroscope_x', 'gyroscope_y', 'gyroscope_z',
        'magnetometer_x', 'magnetometer_y', 'magnetometer_z',
        'orientation_1', 'orientation_2', 'orientation_3', 'orientation_4'
    ]
    
    for position in ['hand', 'chest', 'ankle']:
        for col in imu_columns:
            column_names.append(f'{position}_{col}')
    
    # Load Protocol data
    protocol_path = os.path.join(data_path, "Protocol")
    if os.path.exists(protocol_path):
        print("Loading Protocol data...")
        for subject_file in os.listdir(protocol_path):
            if subject_file.endswith('.dat'):
                file_path = os.path.join(protocol_path, subject_file)
                try:
                    subject_data = pd.read_csv(file_path, sep=' ', header=None, names=column_names)
                    subject_data = subject_data[subject_data['activity_id'] != 0]
                    orientation_cols = [col for col in subject_data.columns if 'orientation' in col]
                    subject_data = subject_data.drop(columns=orientation_cols)
                    feature_cols = [col for col in subject_data.columns if col not in ['timestamp', 'activity_id']]
                    all_data.append(subject_data[feature_cols])
                    all_labels.append(subject_data['activity_id'])
                except Exception as e:
                    print(f"Error loading {file_path}: {e}")
    
    # Load Optional data
    optional_path = os.path.join(data_path, "Optional")
    if os.path.exists(optional_path):
        print("Loading Optional data...")
        for subject_file in os.listdir(optional_path):
            if subject_file.endswith('.dat'):
                file_path = os.path.join(optional_path, subject_file)
                try:
                    subject_data = pd.read_csv(file_path, sep=' ', header=None, names=column_names)
                    subject_data = subject_data[subject_data['activity_id'] != 0]
                    orientation_cols = [col for col in subject_data.columns if 'orientation' in col]
                    subject_data = subject_data.drop(columns=orientation_cols)
                    feature_cols = [col for col in subject_data.columns if col not in ['timestamp', 'activity_id']]
                    all_data.append(subject_data[feature_cols])
                    all_labels.append(subject_data['activity_id'])
                except Exception as e:
                    print(f"Error loading {file_path}: {e}")
    
    X = pd.concat(all_data, ignore_index=True)
    y = pd.concat(all_labels, ignore_index=True)
    
    print(f"Loaded dataset shape: {X.shape}")
    return X, y

# =====================================
# 2. PatchTST Model Architecture
# =====================================
class PatchEmbedding(nn.Module):
    def __init__(self, seq_len, patch_len, d_model, n_vars, dropout=0.1):
        super(PatchEmbedding, self).__init__()
        self.seq_len = seq_len
        self.patch_len = patch_len
        self.d_model = d_model
        self.n_vars = n_vars
        
        # Calculate number of patches
        self.num_patches = (seq_len) // patch_len
        
        # Patch embedding layer - separate for each variable
        self.patch_embedding = nn.Linear(patch_len, d_model)
        
        # Positional encoding
        self.pos_embedding = nn.Parameter(torch.randn(1, self.num_patches * n_vars + 1, d_model))
        
        # CLS token
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model))
        
        # Dropout
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x):
        # x shape: [B, seq_len, n_vars]
        B, seq_len, n_vars = x.shape
        
        # Ensure sequence length is divisible by patch length
        if seq_len % self.patch_len != 0:
            # Pad if necessary
            pad_length = self.patch_len - (seq_len % self.patch_len)
            x = F.pad(x, (0, 0, 0, pad_length))
            seq_len = x.shape[1]
        
        # Reshape into patches
        # [B, seq_len, n_vars] -> [B, num_patches, patch_len, n_vars]
        x = x.reshape(B, self.num_patches, self.patch_len, n_vars)
        
        # Process each variable separately and flatten
        patches_flat = []
        for var_idx in range(n_vars):
            var_patches = x[:, :, :, var_idx]  # [B, num_patches, patch_len]
            var_embedded = self.patch_embedding(var_patches)  # [B, num_patches, d_model]
            patches_flat.append(var_embedded)
        
        # Concatenate all variables
        x = torch.cat(patches_flat, dim=1)  # [B, num_patches * n_vars, d_model]
        
        # Add CLS token
        cls_tokens = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls_tokens, x], dim=1)  # [B, num_patches * n_vars + 1, d_model]
        
        # Add positional encoding
        x = x + self.pos_embedding[:, :x.size(1), :]
        
        return self.dropout(x)

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1):
        super(MultiHeadAttention, self).__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        
        assert self.head_dim * n_heads == d_model, "d_model must be divisible by n_heads"
        
        self.w_q = nn.Linear(d_model, d_model)
        self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model)
        self.w_o = nn.Linear(d_model, d_model)
        
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x, mask=None):
        B, seq_len, d_model = x.shape
        
        # Linear projections
        Q = self.w_q(x).view(B, seq_len, self.n_heads, self.head_dim).transpose(1, 2)
        K = self.w_k(x).view(B, seq_len, self.n_heads, self.head_dim).transpose(1, 2)
        V = self.w_v(x).view(B, seq_len, self.n_heads, self.head_dim).transpose(1, 2)
        
        # Scaled dot-product attention
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.head_dim)
        
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
        
        attention = F.softmax(scores, dim=-1)
        attention = self.dropout(attention)
        
        # Apply attention to values
        out = torch.matmul(attention, V)
        out = out.transpose(1, 2).contiguous().view(B, seq_len, d_model)
        
        # Final linear projection
        out = self.w_o(out)
        return out

class TransformerEncoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff=256, dropout=0.1):
        super(TransformerEncoderLayer, self).__init__()
        self.self_attention = MultiHeadAttention(d_model, n_heads, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        
        # Feed-forward network
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout)
        )
        
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x):
        # Self-attention with residual connection and layer norm
        attn_out = self.self_attention(x)
        x = self.norm1(x + self.dropout(attn_out))
        
        # Feed-forward with residual connection and layer norm
        ffn_out = self.ffn(x)
        x = self.norm2(x + ffn_out)
        
        return x

class PatchTST(nn.Module):
    def __init__(self, input_dim, num_classes, seq_len=100, d_model=64, n_heads=4, 
                 n_layers=3, patch_len=10, dropout=0.1):
        super(PatchTST, self).__init__()
        self.seq_len = seq_len
        self.input_dim = input_dim
        self.d_model = d_model
        self.patch_len = patch_len
        
        # Calculate number of patches
        self.num_patches = seq_len // patch_len
        
        # Patch embedding
        self.patch_embedding = PatchEmbedding(seq_len, patch_len, d_model, input_dim, dropout)
        
        # Transformer encoder layers
        self.encoder_layers = nn.ModuleList([
            TransformerEncoderLayer(d_model, n_heads, d_ff=d_model*2, dropout=dropout)
            for _ in range(n_layers)
        ])
        
        # Layer normalization
        self.norm = nn.LayerNorm(d_model)
        
        # Output classifier (using only CLS token)
        self.classifier = nn.Sequential(
            nn.Linear(d_model, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, num_classes)
        )
        
    def forward(self, x):
        # x shape: [B, seq_len, input_dim]
        B = x.shape[0]
        
        # Patch embedding
        x = self.patch_embedding(x)  # [B, num_patches * input_dim + 1, d_model]
        
        # Transformer encoder
        for encoder_layer in self.encoder_layers:
            x = encoder_layer(x)
        
        x = self.norm(x)
        
        # Use only CLS token for classification
        cls_token = x[:, 0, :]  # [B, d_model]
        
        # Classification
        out = self.classifier(cls_token)
        
        return out

class PAMAP2Dataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# =====================================
# 3. Training Function
# =====================================
def train_patchtst_model(model, train_loader, val_loader, epochs=10):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    
    train_losses = []
    val_accuracies = []
    best_val_acc = 0
    training_start = time.time()
    
    for epoch in range(epochs):
        # Training
        model.train()
        running_loss = 0.0
        correct_train = 0
        total_train = 0
        
        for batch_idx, (data, target) in enumerate(train_loader):
            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, target)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            
            running_loss += loss.item()
            _, predicted = torch.max(output.data, 1)
            total_train += target.size(0)
            correct_train += (predicted == target).sum().item()
            
            if batch_idx % 50 == 0:
                print(f'  Batch {batch_idx}, Loss: {loss.item():.4f}')
        
        scheduler.step()
        
        train_acc = 100 * correct_train / total_train
        avg_loss = running_loss / len(train_loader)
        train_losses.append(avg_loss)
        
        # Validation
        model.eval()
        correct_val = 0
        total_val = 0
        
        with torch.no_grad():
            for data, target in val_loader:
                output = model(data)
                _, predicted = torch.max(output.data, 1)
                total_val += target.size(0)
                correct_val += (predicted == target).sum().item()
        
        val_acc = 100 * correct_val / total_val
        val_accuracies.append(val_acc)
        
        print(f'Epoch {epoch+1}/{epochs}:')
        print(f'  Loss: {avg_loss:.4f}, Train Acc: {train_acc:.2f}%, Val Acc: {val_acc:.2f}%')
        print(f'  Learning Rate: {scheduler.get_last_lr()[0]:.6f}')
        
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), 'best_patchtst_model.pth')
            print(f'  → New best model saved! (Val Acc: {val_acc:.2f}%)')
    
    training_time = time.time() - training_start
    return train_losses, val_accuracies, training_time

# =====================================
# 4. Main Execution
# =====================================
def main():
    print(" Starting PatchTST Model Training on PAMAP2 Dataset")
    
    # Load and preprocess data
    print("\n Loading data...")
    X, y = load_pamap2_data("PAMAP2_Dataset")
    
    # Preprocessing
    X_clean = X.fillna(X.mean())
    constant_cols = X_clean.columns[X_clean.nunique() <= 1]
    if constant_cols.any():
        X_clean = X_clean.drop(columns=constant_cols)
        print(f"Removed {len(constant_cols)} constant columns")
    
    X_array = X_clean.values.astype(np.float32)
    le = LabelEncoder()
    y_encoded = le.fit_transform(y)
    
    print(f"Classes: {np.unique(y_encoded)}")
    
    # Feature scaling
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_array)
    
    # Create sequences
    def create_sequences(X, y, seq_len=100, stride=20):
        X_seq, y_seq = [], []
        for i in range(0, len(X) - seq_len, stride):
            X_seq.append(X[i:i+seq_len])
            sequence_labels = y[i:i+seq_len]
            most_common_label = np.argmax(np.bincount(sequence_labels))
            y_seq.append(most_common_label)
        return np.array(X_seq, dtype=np.float32), np.array(y_seq, dtype=np.int64)
    
    sequence_length = 100
    stride = 20
    X_seq, y_seq = create_sequences(X_scaled, y_encoded, seq_len=sequence_length, stride=stride)
    
    print(f"Sequences created: {X_seq.shape}, Labels: {y_seq.shape}")
    
    # Split data
    X_train, X_temp, y_train, y_temp = train_test_split(X_seq, y_seq, test_size=0.3, stratify=y_seq, random_state=42)
    X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42)
    
    print(f"Data splits - Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")
    
    # Model configuration
    input_dim = X_train.shape[2]
    num_classes = len(np.unique(y_train))
    
    print(f"\n Model Configuration:")
    print(f"Input dimension: {input_dim}")
    print(f"Sequence length: {sequence_length}")
    print(f"Number of classes: {num_classes}")
    
    # Create datasets and loaders
    train_dataset = PAMAP2Dataset(X_train, y_train)
    val_dataset = PAMAP2Dataset(X_val, y_val)
    test_dataset = PAMAP2Dataset(X_test, y_test)
    
    batch_size = 32
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    
    # Initialize PatchTST model with compatible patch length
    patch_len = 10  # 100 sequence length / 10 patches = 10 time steps per patch
    model = PatchTST(
        input_dim=input_dim,
        num_classes=num_classes,
        seq_len=sequence_length,
        d_model=64,
        n_heads=4,
        n_layers=3,
        patch_len=patch_len,
        dropout=0.1
    )
    
    # Calculate parameters
    def count_parameters(model):
        return sum(p.numel() for p in model.parameters() if p.requires_grad)
    
    total_params = count_parameters(model)
    print(f"Model parameters: {total_params:,}")
    print(f"Number of patches: {sequence_length // patch_len}")
    print(f"Total tokens: {1 + (sequence_length // patch_len) * input_dim}")  # CLS + patches * variables
    
    # Test forward pass
    print("\n Testing forward pass...")
    try:
        test_batch = torch.tensor(X_train[:2], dtype=torch.float32)
        print(f"Test batch shape: {test_batch.shape}")
        with torch.no_grad():
            output = model(test_batch)
        print(f"Forward pass successful! Output shape: {output.shape}")
    except Exception as e:
        print(f"Forward pass failed: {e}")
        import traceback
        traceback.print_exc()
        return
    
    # Training
    print(f"\n{'='*60}")
    print(f"Training PatchTST Model")
    print(f"{'='*60}")
    
    train_losses, val_accuracies, training_time = train_patchtst_model(model, train_loader, val_loader, epochs=5)
    
    # =====================================
    # 5. Evaluation and Excel Export
    # =====================================
    print(f"\n{'='*60}")
    print(f"Evaluating PatchTST Model")
    print(f"{'='*60}")
    
    # Load best model
    try:
        model.load_state_dict(torch.load('best_patchtst_model.pth'))
        print("Loaded best model for evaluation")
    except:
        print("Using final model for evaluation")
    
    model.eval()
    
    # Evaluation on test set
    all_predictions, all_probabilities, all_targets = [], [], []
    prediction_start = time.time()
    
    with torch.no_grad():
        for data, target in test_loader:
            output = model(data)
            probabilities = torch.softmax(output, dim=1)
            _, predictions = torch.max(output, 1)
            all_predictions.extend(predictions.numpy())
            all_probabilities.extend(probabilities.numpy())
            all_targets.extend(target.numpy())
    
    total_prediction_time = (time.time() - prediction_start) * 1000
    prediction_time_per_sample = total_prediction_time / len(all_targets)
    
    all_predictions = np.array(all_predictions)
    all_probabilities = np.array(all_probabilities)
    all_targets = np.array(all_targets)
    
    # Calculate metrics
    accuracy = accuracy_score(all_targets, all_predictions)
    f1 = f1_score(all_targets, all_predictions, average='weighted')
    precision = precision_score(all_targets, all_predictions, average='weighted')
    recall = recall_score(all_targets, all_predictions, average='weighted')
    
    try:
        roc_auc = roc_auc_score(all_targets, all_probabilities, multi_class='ovr', average='weighted')
    except:
        roc_auc = 0.0
    
    mae = mean_absolute_error(all_targets, all_predictions)
    mse = mean_squared_error(all_targets, all_predictions)
    r2 = r2_score(all_targets, all_predictions)
    
    # Throughput
    test_batch = torch.tensor(X_test[:batch_size], dtype=torch.float32)
    start_time = time.time()
    with torch.no_grad():
        _ = model(test_batch)
    batch_time = (time.time() - start_time) * 1000
    throughput = (batch_size / batch_time) * 1000
    
    # Memory usage
    process = psutil.Process(os.getpid())
    memory_usage = process.memory_info().rss / (1024 ** 2)
    
    # FLOPs estimation for PatchTST
    def estimate_patchtst_flops(input_shape):
        batch_size, seq_len, input_dim = input_shape
        d_model = 64
        n_heads = 4
        n_layers = 3
        patch_len = 10
        
        # Calculate number of patches and tokens
        num_patches = seq_len // patch_len
        total_tokens = 1 + num_patches * input_dim  # CLS + patches * variables
        
        # Patch embedding FLOPs
        patch_embedding_flops = 2 * patch_len * d_model * num_patches * input_dim
        
        # Transformer FLOPs per layer
        # Self-attention: 4 * d_model^2 * total_tokens + 2 * d_model * total_tokens^2
        attention_flops_per_layer = (4 * d_model * d_model * total_tokens + 
                                   2 * d_model * total_tokens * total_tokens)
        
        # FFN FLOPs: 2 * d_model * d_ff * total_tokens (d_ff = 2 * d_model)
        ffn_flops_per_layer = 2 * d_model * (2 * d_model) * total_tokens
        
        # Total transformer FLOPs
        transformer_flops = n_layers * (attention_flops_per_layer + ffn_flops_per_layer)
        
        # Classifier FLOPs
        classifier_flops = 2 * d_model * 128 + 2 * 128 * 64 + 2 * 64 * num_classes
        
        total_flops = patch_embedding_flops + transformer_flops + classifier_flops
        return total_flops
    
    flops = estimate_patchtst_flops((1, sequence_length, input_dim))
    
    # Calculate latency
    latency_per_sample = 1 / (throughput / 1000) * 1000 if throughput > 0 else 0
    
    # =====================================
    # 6. Save Results to Excel
    # =====================================
    print("\n💾 Saving results to Excel...")
    
    # Create comprehensive results
    results_data = {
        'Metric_Type': ['Performance', 'Performance', 'Performance', 'Performance', 'Performance', 
                       'Performance', 'Performance', 'Performance', 'Model', 'Model', 'Model', 
                       'Model', 'Model', 'Model', 'Model', 'Model', 'Model', 'Model', 'Dataset'],
        'Metric_Name': ['Accuracy', 'F1_Score', 'Precision', 'Recall', 'ROC_AUC', 'MAE', 'MSE', 'R2_Score',
                       'Parameters', 'Model_Size_MB', 'Training_Time_seconds', 'Prediction_Time_ms', 
                       'Throughput_samples_sec', 'Latency_ms', 'Sequence_Length', 'Features', 'FLOPs', 
                       'Memory_Usage_MB', 'Test_Samples'],
        'Value': [accuracy, f1, precision, recall, roc_auc, mae, mse, r2,
                 total_params, total_params * 4 / (1024**2), training_time, 
                 prediction_time_per_sample, throughput, latency_per_sample, sequence_length, 
                 input_dim, flops, memory_usage, len(X_test)],
        'Description': [
            'Classification accuracy', 'Weighted F1-score', 'Weighted precision', 'Weighted recall',
            'Area under ROC curve', 'Mean Absolute Error', 'Mean Squared Error', 'R-squared score',
            'Total trainable parameters', 'Model size in megabytes', 'Time taken for training', 
            'Prediction time per sample', 'Samples processed per second', 'Latency per prediction',
            'Sequence length for time series', 'Number of input features', 'Floating point operations',
            'Memory usage during inference', 'Number of test samples'
        ]
    }
    
    results_df = pd.DataFrame(results_data)
    
    # Create detailed classification report
    classification_rep = classification_report(all_targets, all_predictions, output_dict=True)
    class_report_df = pd.DataFrame(classification_rep).transpose()
    
    # Create confusion matrix data
    cm = confusion_matrix(all_targets, all_predictions)
    cm_df = pd.DataFrame(cm, index=[f'True_{i}' for i in range(cm.shape[0])], 
                        columns=[f'Pred_{i}' for i in range(cm.shape[1])])
    
    # Create training history
    training_history = pd.DataFrame({
        'Epoch': range(1, len(train_losses) + 1),
        'Training_Loss': train_losses,
        'Validation_Accuracy': val_accuracies
    })
    
    # Save to Excel with multiple sheets
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = f'PatchTST_PAMAP2_Results_{timestamp}.xlsx'
    
    with pd.ExcelWriter(filename, engine='openpyxl') as writer:
        results_df.to_excel(writer, sheet_name='Summary_Metrics', index=False)
        class_report_df.to_excel(writer, sheet_name='Detailed_Classification_Report')
        cm_df.to_excel(writer, sheet_name='Confusion_Matrix')
        training_history.to_excel(writer, sheet_name='Training_History', index=False)
    
    # =====================================
    # 7. Display Final Results
    # =====================================
    print(f"\n{'='*80}")
    print(f"🎯 PATCHTST MODEL RESULTS - PAMAP2 DATASET")
    print(f"{'='*80}")
    
    print(f"\n📊 PERFORMANCE METRICS:")
    print(f"Accuracy:       {accuracy:.4f}")
    print(f"F1-Score:       {f1:.4f}")
    print(f"Precision:      {precision:.4f}")
    print(f"Recall:         {recall:.4f}")
    print(f"ROC-AUC:        {roc_auc:.4f}")
    print(f"MAE:            {mae:.4f}")
    print(f"MSE:            {mse:.4f}")
    print(f"R² Score:       {r2:.4f}")
    
    print(f"\n⚡ EFFICIENCY METRICS:")
    print(f"Parameters:     {total_params:,}")
    print(f"Model Size:     {total_params * 4 / (1024**2):.2f} MB")
    print(f"Training Time:  {training_time:.2f} seconds")
    print(f"Prediction Time:{prediction_time_per_sample:.4f} ms")
    print(f"Throughput:     {throughput:.2f} samples/sec")
    print(f"Latency:        {latency_per_sample:.4f} ms")
    print(f"Sequence Length:{sequence_length}")
    print(f"Features:       {input_dim}")
    print(f"FLOPs:          {flops:,}")
    print(f"Memory Usage:   {memory_usage:.2f} MB")
    
    print(f"\n🔧 MODEL ARCHITECTURE:")
    print(f"Patch Length:   {patch_len}")
    print(f"Number of Patches: {sequence_length // patch_len}")
    print(f"Total Tokens:   {1 + (sequence_length // patch_len) * input_dim}")
    print(f"Number of Heads: 4")
    print(f"Transformer Layers: 3")
    print(f"Hidden Dimension: 64")
    
    print(f"\n💾 RESULTS SAVED TO: {filename}")
    print(f"\n✅ PatchTST Model Training and Evaluation Completed!")

if __name__ == "__main__":
    main()

Using CPU for stability
🚀 Starting PatchTST Model Training on PAMAP2 Dataset

📊 Loading data...
Loading PAMAP2 dataset from local files...
Loading Protocol data...
Loading Optional data...
Loaded dataset shape: (2724953, 40)
Classes: [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17]
Sequences created: (136243, 100, 40), Labels: (136243,)
Data splits - Train: (95370, 100, 40), Val: (20436, 100, 40), Test: (20437, 100, 40)

⚙️ Model Configuration:
Input dimension: 40
Sequence length: 100
Number of classes: 18
Model parameters: 144,722
Number of patches: 10
Total tokens: 401

🧪 Testing forward pass...
Test batch shape: torch.Size([2, 100, 40])
Forward pass successful! Output shape: torch.Size([2, 18])

Training PatchTST Model
  Batch 0, Loss: 2.8938
  Batch 50, Loss: 2.7203
  Batch 100, Loss: 2.2731
  Batch 150, Loss: 1.5876
  Batch 200, Loss: 2.0513
  Batch 250, Loss: 1.3535
  Batch 300, Loss: 1.8462
  Batch 350, Loss: 1.4491
  Batch 400, Loss: 1.6909
  Batch 450, Loss: 1.2077
  Ba